# Неделя 3 — Признаки и CatBoost

Задание: `docs/week3_baseline.md`


In [1]:
#ЗАГРУЗКА ДАТАСЕТА И ИМПОРТ
import sys
import pandas as pd
import numpy as np
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from catboost import CatBoostClassifier, Pool

sys.path.append('..')
from src.validation import evaluate, split_by_time

# Загружаем датасет
dataset = pd.read_parquet('../data/processed/dataset.parquet')
print("Размер датасета:", dataset.shape)

Размер датасета: (257631, 22)


In [2]:
#РАЗБИЕНИЕ ПО ВРЕМЕНИ
train, valid, test = split_by_time(dataset)
print(f"Train: {len(train)}, Valid: {len(valid)}, Test: {len(test)}")
print(f"Доля оттока — train: {train['target'].mean():.4f}, test: {test['target'].mean():.4f}")

#определяем колонки
drop_cols = ['target', 'client_id', 'snapshot_date']
cat_cols = ['segment', 'product', 'region']
num_cols = [c for c in test.columns if c not in drop_cols + cat_cols]

#заполняем пропуски в категориальных признаках (требование CatBoost)
for col in cat_cols:
    train[col] = train[col].fillna('unknown')
    valid[col] = valid[col].fillna('unknown')
    test[col] = test[col].fillna('unknown')

X_train, y_train = train.drop(columns=drop_cols), train['target']
X_valid, y_valid = valid.drop(columns=drop_cols), valid['target']
X_test, y_test = test.drop(columns=drop_cols), test['target']

#список для хранения метрик всех моделей
metrics_list = []

Train: 66253, Valid: 35788, Test: 40265
Доля оттока — train: 0.0762, test: 0.1379


In [3]:
print("\nBaseline 0: Правило")

#проверяем точное название колонки
feature_name = 'revenue_last_to_mean' if 'revenue_last_to_mean' in test.columns else 'rev_last_to_mean'

#минус обязателен: чем сильнее упал платёж, тем выше риск
y_pred_rule = -X_test[feature_name]
metrics_list.append(evaluate(y_test, y_pred_rule, 'Правило'))


Baseline 0: Правило
--- Правило ---
ROC-AUC: 0.8495
PR-AUC: 0.6476
Precision@10%: 0.7501
Lift@10%: 5.44x
Доля оттока: 0.1379


In [4]:
print("\nBaseline 1: Логистическая регрессия")

preprocessor = ColumnTransformer(
    transformers=[
        ('num', Pipeline([('imputer', SimpleImputer(strategy='median')), 
                          ('scaler', StandardScaler())]), num_cols),
        ('cat', Pipeline([('encoder', OneHotEncoder(handle_unknown='ignore', sparse_output=False))]), cat_cols)
    ])

logreg_pipeline = Pipeline(steps=[
    ('preprocessor', preprocessor),
    ('classifier', LogisticRegression(max_iter=1000, class_weight='balanced', random_state=42))
])

logreg_pipeline.fit(X_train, y_train)
y_pred_logreg = logreg_pipeline.predict_proba(X_test)[:, 1]
metrics_list.append(evaluate(y_test, y_pred_logreg, 'Логистическая регрессия'))


Baseline 1: Логистическая регрессия
--- Логистическая регрессия ---
ROC-AUC: 0.8799
PR-AUC: 0.7059
Precision@10%: 0.7655
Lift@10%: 5.55x
Доля оттока: 0.1379


## 1. Построение признаков по группам

TODO: реализуйте make_features() в src/features.py (платежи, динамика, объём отношений, сигналы боли).


## 2. Обучение CatBoost

TODO: CatBoostClassifier с early stopping на eval-срезе по времени (не случайном!).


In [5]:

#cоздаем пулы для CatBoost
train_pool = Pool(
    data=X_train,
    label=y_train,
    cat_features=cat_cols
)

valid_pool = Pool(
    data=X_valid,
    label=y_valid,
    cat_features=cat_cols
)

#инициализируем модель
model_cb = CatBoostClassifier(
    iterations=2000,
    learning_rate=0.05,
    depth=6,
    eval_metric='AUC',
    random_seed=42,
    verbose=200,
    early_stopping_rounds=100  # Остановится, если 100 итераций нет улучшения на valid
)

#обучаем модель с early stopping строго на валидационном снапшоте (2025-01)!
model_cb.fit(train_pool, eval_set=valid_pool)

#оцениваем на тесте (используем тест ровно один раз!)
y_pred_cb = model_cb.predict_proba(X_test)[:, 1]
metrics_list.append(evaluate(y_test, y_pred_cb, 'CatBoost'))

0:	test: 0.8347909	best: 0.8347909 (0)	total: 179ms	remaining: 5m 58s
200:	test: 0.8921978	best: 0.8921978 (200)	total: 18.5s	remaining: 2m 45s
400:	test: 0.8933781	best: 0.8936152 (360)	total: 36.2s	remaining: 2m 24s
Stopped by overfitting detector  (100 iterations wait)

bestTest = 0.8936151633
bestIteration = 360

Shrink model to first 361 iterations.
--- CatBoost ---
ROC-AUC: 0.9147
PR-AUC: 0.8206
Precision@10%: 0.9175
Lift@10%: 6.65x
Доля оттока: 0.1379


## 3. Итоговая таблица метрик: правило vs логрегрессия vs CatBoost


In [6]:
print("\n" + "="*60)
print("ИТОГОВАЯ СВОДНАЯ ТАБЛИЦА МЕТРИК НА ТЕСТЕ")
print("="*60)

#преобразуем список словарей в красивый DataFrame
results_df = pd.DataFrame(metrics_list)
results_df = results_df[['label', 'roc_auc', 'pr_auc', 'precision_at_10', 'lift_at_10']]
results_df.columns = ['Модель', 'ROC-AUC', 'PR-AUC', 'Precision@10%', 'Lift@10%']

#форматируем числа для красоты
results_df['ROC-AUC'] = results_df['ROC-AUC'].apply(lambda x: f"{x:.4f}")
results_df['PR-AUC'] = results_df['PR-AUC'].apply(lambda x: f"{x:.4f}")
results_df['Precision@10%'] = results_df['Precision@10%'].apply(lambda x: f"{x:.4f}")
results_df['Lift@10%'] = results_df['Lift@10%'].apply(lambda x: f"{x:.2f}x")

print(results_df.to_string(index=False))


ИТОГОВАЯ СВОДНАЯ ТАБЛИЦА МЕТРИК НА ТЕСТЕ
                 Модель ROC-AUC PR-AUC Precision@10% Lift@10%
                Правило  0.8495 0.6476        0.7501    5.44x
Логистическая регрессия  0.8799 0.7059        0.7655    5.55x
               CatBoost  0.9147 0.8206        0.9175    6.65x


## 4. Вывод на языке бизнеса

TODO: одна фраза вида 'в топ-10% клиентов по риску попадает X% всех будущих уходов, lift = Yx'.


In [ ]:
#берем метрики лучшей модели (обычно это последняя, CatBoost)
best_model_metrics = metrics_list[-1]
lift_val = best_model_metrics['lift_at_10']
prec_val = best_model_metrics['precision_at_10'] * 100  # переводим в проценты

print("ВЫВОД НА ЯЗЫКЕ БИЗНЕСА:")
print(f"В топ-10% клиентов по риску попадает {prec_val:.1f}% всех будущих уходов, "
      f"lift = {lift_val:.2f}x. Обзвонив 10% базы, отдел удержания застанет "
      f"{prec_val:.1f}% уходов вместо 10% при случайном выборе.")


ВЫВОД НА ЯЗЫКЕ БИЗНЕСА:
В топ-10% клиентов по риску попадает 91.8% всех будущих уходов, lift = 6.65x. Обзвонив 10% базы, отдел удержания застанет 91.8% уходов вместо 10% при случайном выборе.
